# Task 2: Exploratory Data Analysis & Data Characterization

Characterize TIM Milan mobile internet traffic using the optimized Parquet from Task 1. Figures below support the report sections (PDF, time series, stationarity, decomposition, ACF/PACF, spatial patterns, anomalies).

In [ ]:
# Imports for data, plots, and time-series statistics
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

sns.set_theme(style="whitegrid")

In [ ]:
# Paths and study window (two calendar months: Nov–Dec 2013)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "milan_internet_traffic.parquet"

TWO_MONTH_START = "2013-11-01"
TWO_MONTH_END = "2014-01-01"  # exclusive upper bound
FIRST_TWO_WEEKS_END = "2013-11-15"  # Nov 1–14 inclusive

# Areas required in Task 2 II (+ highest-traffic id discovered from data)
SQUARE_4159 = 4159
SQUARE_4556 = 4556

FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
# Load processed data (fast Parquet from Task 1)
traffic = pd.read_parquet(PARQUET_PATH)
traffic = traffic[(traffic["time"] >= TWO_MONTH_START) & (traffic["time"] < TWO_MONTH_END)]
print(f"Rows in two-month window: {len(traffic):,}")
print(f"Time span: {traffic['time'].min()} → {traffic['time'].max()}")

## II.1 — Probability density of total two-month traffic per area

Each of the 10,000 grid cells contributes one sample: the **sum of internet traffic** over Nov–Dec 2013.

In [ ]:
# Total traffic per square over the two-month period (10,000 values)
area_totals = traffic.groupby("square_id")["internet"].sum()
print(f"Areas: {len(area_totals)}  |  min={area_totals.min():.0f}  max={area_totals.max():.0f}")

In [ ]:
# KDE estimates the PDF; log-x helps see the heavy tail
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(area_totals, bins=60, stat="density", kde=True, color="steelblue", ax=ax)
ax.set_xscale("log")
ax.set_xlabel("Total two-month internet traffic (log scale)")
ax.set_ylabel("Density")
ax.set_title("PDF of total traffic across 10,000 Milan grid cells")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_area_total_pdf.png", dpi=150)
plt.show()

**Interpretation (for report):** The distribution is **strongly right-skewed** — a few central/high-activity cells dominate total traffic, while many suburban cells stay low. This spatial heterogeneity matches a dense urban core plus quieter periphery.

## II.2 — Time series: first two weeks at three areas

(i) highest total traffic, (ii) square 4159, (iii) square 4556.

In [ ]:
# Identify the busiest cell in the two-month window
SQUARE_HIGHEST = int(area_totals.idxmax())
FOCUS_SQUARES = [SQUARE_HIGHEST, SQUARE_4159, SQUARE_4556]
print("Focus squares:", FOCUS_SQUARES)

In [ ]:
def series_for_square(df: pd.DataFrame, square_id: int) -> pd.Series:
    """Regular 10-minute series for one square (missing slots filled with 0)."""
    s = (
        df.loc[df["square_id"] == square_id, ["time", "internet"]]
        .set_index("time")["internet"]
        .sort_index()
    )
    full_index = pd.date_range(s.index.min(), s.index.max(), freq="10min")
    return s.reindex(full_index, fill_value=0)

In [ ]:
# Subset: first two weeks of November 2013
mask_two_weeks = (traffic["time"] >= TWO_MONTH_START) & (traffic["time"] < FIRST_TWO_WEEKS_END)
traffic_2w = traffic.loc[mask_two_weeks]

In [ ]:
# Plot three areas side by side
labels = {
    SQUARE_HIGHEST: f"Highest traffic (id {SQUARE_HIGHEST})",
    SQUARE_4159: "Square 4159",
    SQUARE_4556: "Square 4556",
}

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, sq in zip(axes, FOCUS_SQUARES):
    ts = series_for_square(traffic_2w, sq)
    ax.plot(ts.index, ts.values, linewidth=0.8)
    ax.set_ylabel("Traffic")
    ax.set_title(labels[sq])
axes[-1].set_xlabel("Time")
fig.suptitle("Network traffic — first two weeks (Nov 2013)", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_three_areas_two_weeks.png", dpi=150)
plt.show()

**Interpretation:** All three series show **strong daily seasonality** (day/night cycles) and **lower weekend activity**. The highest-traffic cell has larger peaks (likely central Milan); 4159 and 4556 differ in baseline level and peak timing, suggesting different land-use mixes nearby.

## II.3 — Stationarity: rolling statistics & Augmented Dickey–Fuller test

Analysis on the **highest-traffic** area (full two months); ADF also reported for the other two squares.

In [ ]:
# Full two-month series for the busiest square
ts_main = series_for_square(traffic, SQUARE_HIGHEST)
window = 144  # 1 day = 144 ten-minute intervals

In [ ]:
# Rolling mean and std (non-stationary series often drift in mean/variance)
roll_mean = ts_main.rolling(window).mean()
roll_std = ts_main.rolling(window).std()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts_main.index, ts_main.values, label="Traffic", alpha=0.5, linewidth=0.6)
ax.plot(roll_mean.index, roll_mean.values, label=f"Rolling mean ({window} pts)", color="red")
ax.plot(roll_std.index, roll_std.values, label=f"Rolling std ({window} pts)", color="green")
ax.legend()
ax.set_title(f"Rolling statistics — square {SQUARE_HIGHEST}")
ax.set_xlabel("Time")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_rolling_stats.png", dpi=150)
plt.show()

In [ ]:
# ADF test: H0 = unit root (non-stationary). Small p-value → reject H0
def run_adf(series: pd.Series) -> dict:
    stat, pval, _, _, crit, _ = adfuller(series.values, autolag="AIC")
    return {"ADF statistic": stat, "p-value": pval, "1% crit": crit["1%"]}

adf_rows = []
for sq in FOCUS_SQUARES:
    s = series_for_square(traffic, sq)
    row = {"square_id": sq, **run_adf(s)}
    adf_rows.append(row)

adf_table = pd.DataFrame(adf_rows)
display(adf_table)

**Interpretation:** Rolling mean/std **change over time** (weekends, holidays), so the raw series is **not stationary**. ADF usually **fails to reject** the unit-root null at α=0.05; differencing or seasonal detrending is appropriate before ARIMA-style models (Task 3).

## II.4 — Decomposition (trend, seasonal, residual)

Additive decomposition with **daily period** = 144 intervals per day.

In [ ]:
# Need equally spaced observations for decomposition
period = 144
decomp = seasonal_decompose(ts_main, model="additive", period=period, extrapolate_trend="freq")

In [ ]:
fig = decomp.plot()
fig.set_size_inches(10, 8)
fig.suptitle(f"Additive decomposition — square {SQUARE_HIGHEST}", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_decomposition.png", dpi=150)
plt.show()

**Interpretation:** **Seasonal** component captures the day/night cycle; **trend** slowly shifts with long-term level changes; **residual** spikes may be anomalies (events, outages). A **weekly period** (1008) could be explored as extension.

## II.5 — Autocorrelation (ACF) and partial autocorrelation (PACF)

In [ ]:
# Use a subset for readable plots (first 14 days)
ts_14d = ts_main.loc[: ts_main.index[0] + pd.Timedelta(days=14)]
lags = 200

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_acf(ts_14d, lags=lags, ax=axes[0])
axes[0].set_title("ACF")
plot_pacf(ts_14d, lags=lags, ax=axes[1], method="ywm")
axes[1].set_title("PACF")
fig.suptitle(f"Square {SQUARE_HIGHEST} — first 14 days")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_acf_pacf.png", dpi=150)
plt.show()

**Interpretation:** ACF shows **slow decay** and **spikes at multiples of 144 lags** (daily seasonality). PACF has sharp cuts at short lags — typical of seasonal AR structure; motivates SARIMA or sequence models with ~144-step history.

## II.6 — Spatial analysis: heatmap on 100×100 grid

Square ids map to grid cells: `row = (id-1)//100`, `col = (id-1)%100` (TIM Milan tessellation).

In [ ]:
# Build 100×100 matrix of total two-month traffic
grid = np.zeros((100, 100), dtype=np.float32)
for sq_id, total in area_totals.items():
    r, c = divmod(int(sq_id) - 1, 100)
    grid[r, c] = total

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(grid, origin="lower", cmap="YlOrRd")
ax.set_title("Spatial heatmap — total two-month internet traffic")
ax.set_xlabel("Grid column")
ax.set_ylabel("Grid row")
plt.colorbar(im, ax=ax, label="Total traffic")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_spatial_heatmap.png", dpi=150)
plt.show()

**Interpretation:** Hot spots cluster in the **city center** (bright core); peripheral cells are cooler. This aligns with PDF skewness and land-use (business/residential vs suburbs).

## II.7 — Anomalies, outliers, and unusual behaviour

In [ ]:
# Global z-score on 10-minute values flags extreme intervals
z = stats.zscore(traffic["internet"].astype(float))
traffic["z"] = z
outliers = traffic.loc[traffic["z"].abs() > 5, ["square_id", "time", "internet", "z"]]
print(f"Extreme intervals (|z|>5): {len(outliers):,}")
display(outliers.head(10))

In [ ]:
# Example: largest single-interval spike in the two-month window
top = traffic.nlargest(5, "internet")[["square_id", "time", "internet"]]
display(top)

**Interpretation (for report):**
- **Sparse zeros:** Many intervals are zero (no CDR activity) — especially low-traffic cells.
- **Positive spikes:** Short bursts may reflect events, cell congestion, or measurement noise in the CDR proxy.
- **Holiday dips:** Christmas/New Year weeks show lower city-wide activity (visible in rolling mean).
- **Edge timestamps:** First/last day files can include boundary effects; check when interpreting single intervals.

---

**Task 3 prep:** Save focus square ids for forecasting (Dec 16–22 test week).

| Role | square_id |
|------|-----------|
| Highest traffic | (printed above) |
| Fixed | 4159, 4556 |

In [ ]:
# Export ids for Task 3 notebooks
meta = pd.DataFrame(
    {
        "label": ["highest", "square_4159", "square_4556"],
        "square_id": [SQUARE_HIGHEST, SQUARE_4159, SQUARE_4556],
    }
)
out = PROJECT_ROOT / "data" / "processed" / "task2_focus_squares.csv"
meta.to_csv(out, index=False)
print(f"Saved {out}")
display(meta)